# BioSkills Lab — Chapter 10
## Variational Autoencoders on TCGA
[![BioSkills Lab](https://img.shields.io/badge/BioSkills-Lab-38bdf8)](https://bioskillslab.dev)

> **GPU recommended** — Runtime > Change runtime type > T4 GPU

> **Note:** The full experiment in the lesson uses the complete TCGA dataset (11,000 samples, 20,000 genes) requiring 32GB+ RAM. This notebook uses a smaller subset compatible with **free-tier Google Colab**. To reproduce the full results, run on a local machine with sufficient RAM or a compute cluster.

### What you will do
Build a **Variational Autoencoder (VAE)** and compare its latent space to the standard autoencoder from Chapter 9.

**The key difference from a standard autoencoder:** Instead of mapping each sample to a single point in latent space, a VAE maps each sample to a **probability distribution** (mean + variance). This forces the latent space to be continuous and structured.

**Why this matters in cancer biology:** A continuous latent space lets you discover the spectrum of disease states between cancer subtypes, not just discrete categories.

In [ ]:
!pip install -q torch numpy pandas matplotlib scikit-learn requests

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
print(f'GPU: {torch.cuda.is_available()}')

### Load data
Re-uses Chapter 7 variables if available, otherwise fetches from cBioPortal.

In [ ]:
try:
    _ = X_train_pca
    print('Chapter 7 data already loaded')
except NameError:
    BASE = 'https://www.cbioportal.org/api'
    STUDIES = {'BRCA':'brca_tcga','LUAD':'luad_tcga','KIRC':'kirc_tcga','PRAD':'prad_tcga','COAD':'coad_tcga'}
    GENE_IDS = [672,675,7157,4609,5728,1029,595,896,2064,2065,1956,3320,4292,5594,5595,5604,5605,207,208,
                10000,4893,4914,5290,6662,6663,3845,4763,4764,7422,7424,7525,7528,1499,4005,7409,8503,
                9020,2099,2100,2263,2534,3169,3236,4254,4255,5156,5163,5241,5579,6009,6196,7163,7490,
                7849,8379,9054,9500,10257,10319,10488,11200,23533,51176,55294,1026,1027,1028,1030,1031,
                1032,1869,1870,4616,4619,8317,8318,701,994,999,1000,4193,4831,5925,6237,7153,7538]
    all_frames = []
    for cancer, study in STUDIES.items():
        print(f'  Fetching {cancer}...', end=' ')
        try:
            r = requests.get(f'{BASE}/studies/{study}/samples', params={'pageSize':150}, timeout=30)
            samples = [s['sampleId'] for s in r.json()[:150]]
            r2 = requests.post(f'{BASE}/molecular-profiles/{study}_rna_seq_v2_mrna/molecular-data/fetch',
                               params={'projection':'SUMMARY'},
                               json={'sampleIds':samples,'entrezGeneIds':GENE_IDS}, timeout=120)
            if r2.status_code != 200: print('failed'); continue
            df = pd.DataFrame([{'sample':d['sampleId'],'gene':d['entrezGeneId'],'value':d['value']}
                               for d in r2.json() if d.get('value') is not None])
            pivot = df.pivot_table(index='sample',columns='gene',values='value',aggfunc='first')
            pivot['cancer_type'] = cancer; all_frames.append(pivot)
            print(f'{len(pivot)} samples')
        except Exception as e: print(f'error: {e}')
    if not all_frames:
        np.random.seed(42); n,g=150,500
        blocks,labs=[],[]
        for i,ct in enumerate(STUDIES.keys()):
            X=np.random.randn(n,g)*0.5; X[:,i*100:(i+1)*100]+=3
            blocks.append(X); labs.extend([ct]*n)
        expr=pd.DataFrame(np.vstack(blocks),columns=[f'g{i}' for i in range(g)]); expr['cancer_type']=labs
    else: expr=pd.concat(all_frames)
    y_raw=expr['cancer_type'].values
    X_raw=np.nan_to_num(expr.drop(columns=['cancer_type']).values.astype(np.float32))
    le=LabelEncoder(); y_enc=le.fit_transform(y_raw)
    X_train,X_test,y_train,y_test=train_test_split(X_raw,y_enc,test_size=0.2,random_state=42,stratify=y_enc)
    scaler=StandardScaler()
    X_train_s=scaler.fit_transform(X_train); X_test_s=scaler.transform(X_test)
    pca=PCA(n_components=min(50,X_train_s.shape[1]-1))
    X_train_pca=pca.fit_transform(X_train_s); X_test_pca=pca.transform(X_test_s)
    print(f'Data ready. Shape: {X_train_pca.shape}')

### VAE architecture and loss

The encoder outputs **mu** (mean) and **log_var** (log variance). A sample z is drawn using the **reparameterisation trick**: `z = mu + epsilon * std`. This makes sampling differentiable.

The loss = **reconstruction** (MSE) + **KL divergence** (keeps distributions close to N(0,1)). The KL term produces the continuous, structured latent space.

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        self.encoder    = nn.Sequential(nn.Linear(input_dim,hidden_dim),nn.ReLU(),nn.Linear(hidden_dim,hidden_dim//2),nn.ReLU())
        self.fc_mu      = nn.Linear(hidden_dim//2, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim//2, latent_dim)
        self.decoder    = nn.Sequential(nn.Linear(latent_dim,hidden_dim//2),nn.ReLU(),nn.Linear(hidden_dim//2,hidden_dim),nn.ReLU(),nn.Linear(hidden_dim,input_dim))
    def encode(self, x): h=self.encoder(x); return self.fc_mu(h), self.fc_log_var(h)
    def reparameterise(self, mu, lv): return mu + torch.randn_like(mu)*torch.exp(0.5*lv)
    def decode(self, z): return self.decoder(z)
    def forward(self, x): mu,lv=self.encode(x); return self.decode(self.reparameterise(mu,lv)), mu, lv

def vae_loss(xr, x, mu, lv, beta=1.0):
    recon = F.mse_loss(xr, x, reduction='sum')
    kl    = -0.5 * beta * torch.sum(1 + lv - mu.pow(2) - lv.exp())
    return recon + kl

### Train the VAE

**What to expect:** Loss decreases over 100 epochs. ~1-2 minutes on CPU, ~20 seconds on GPU.

In [ ]:
vae = VAE(X_train_pca.shape[1], 256, 16)
opt = optim.Adam(vae.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(torch.FloatTensor(X_train_pca)), batch_size=128, shuffle=True)
losses = []
for epoch in range(100):
    vae.train(); total=0
    for (x,) in loader:
        opt.zero_grad(); xr,mu,lv=vae(x); loss=vae_loss(xr,x,mu,lv)
        loss.backward(); opt.step(); total+=loss.item()
    losses.append(total/len(loader))
    if (epoch+1)%25==0: print(f'Epoch {epoch+1}: loss={losses[-1]:.1f}')

plt.figure(figsize=(7,3))
plt.plot(losses, color='#a78bfa')
plt.xlabel('Epoch'); plt.ylabel('VAE Loss (recon + KL)')
plt.title('VAE Training Loss')
plt.tight_layout(); plt.show()

### Visualise the VAE latent space

We encode test samples and project the 16D latent vectors to 2D with PCA.

**How to interpret:** Well-separated clusters = VAE learned meaningful representations. Smoother transitions than Chapter 7 PCA = the latent space is more continuous. Compare the two plots — is the VAE latent space better organised?

In [ ]:
vae.eval()
with torch.no_grad():
    mu_test,_=vae.encode(torch.FloatTensor(X_test_pca))
from sklearn.decomposition import PCA as PCA2D
Z_2d=PCA2D(n_components=2).fit_transform(mu_test.numpy())

plt.figure(figsize=(10,7))
colors=['#f97316','#38bdf8','#4ade80','#a78bfa','#f43f5e']
for i,ct in enumerate(le.classes_):
    m=y_test==i
    plt.scatter(Z_2d[m,0],Z_2d[m,1],label=ct,alpha=0.6,s=25,color=colors[i])
plt.xlabel('Latent PC1'); plt.ylabel('Latent PC2')
plt.title('VAE Latent Space — Test Set (projected to 2D)')
plt.legend(fontsize=9, title='Cancer type')
plt.tight_layout(); plt.show()
print('\nVAE training complete. In Chapter 11 we move from bulk RNA-seq to single-cell data with scVI.')